In [1]:
import gymnasium as gym
import numpy as np

class AntiJammingEnv(gym.Env):
    def __init__(self, num_freqs=10, num_powers=5, jammer_type="random"):
        super(AntiJammingEnv, self).__init__()
        
        self.num_freqs = num_freqs
        self.num_powers = num_powers
        self.jammer_type = jammer_type
        
        # Define action space (freq × power combinations)
        self.action_space = gym.spaces.Discrete(num_freqs * num_powers)
        
        # State = PSD (vector of size num_freqs)
        self.observation_space = gym.spaces.Box(
            low=0, high=1, shape=(num_freqs,), dtype=np.float32
        )
        
        self.prev_freq = None
        self.state = None

    def reset(self):
        self.state = np.random.rand(self.num_freqs) * 0.1  # background noise
        self.prev_freq = None
        return self.state

    def step(self, action):
        freq = action // self.num_powers
        power = action % self.num_powers + 1  # scale 1..num_powers
        
        # --- Jamming model (simplified random jammer here) ---
        self.state = np.random.rand(self.num_freqs) * 0.1  # Background noise

        jammed_freq = np.random.randint(0, self.num_freqs)  # Randomly choose a frequency to jam
        self.state[jammed_freq] += 1.0  # strong interference on that channel



        # --- Calculate reward ---
        noise_at_chosen = self.state[freq]
        snr = power / (noise_at_chosen + 1e-6) # includes jamming + noise if freq was jammed
        crc_success = 1 if snr > 1.5 else 0  # threshold for success
        
        # -- Switching penalty ---
        penalty = 0
        if self.prev_freq is not None and self.prev_freq != freq:
            penalty = 0.2  # switching penalty
        

        # --- Reward calculation ---
        reward = crc_success * (np.log(1 + snr) / power - penalty)
        

        # Update previous frequency
        self.prev_freq = freq       #used for switching penalty
        
        done = False  # could end after K timesteps if you want
        info = {}
        
        return self.state, reward, done, info


In [2]:
from stable_baselines3 import DQN

env = AntiJammingEnv()
model = DQN("MlpPolicy", env, verbose=1)
model.learn(total_timesteps=10000, log_interval=4)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


TypeError: AntiJammingEnv.reset() got an unexpected keyword argument 'seed'